### **Dataset Engineering**

Cleaning the following datasets:
1. Infrastructure Dataset 
2. Accidents Dataset

In [121]:
import pandas as pd

In [122]:
df = pd.read_csv('C:/Users/Horia/Desktop/KUL/Modern Data Analytics/cycling infrastructure (2024).csv')
be_infrastructure = df[df['Country'] == "BE"].copy()
be_infrastructure = be_infrastructure[be_infrastructure['NUTS_ID'].str.startswith('BE2')]

#### **Building the Infrastructure Deficit Metric for each city**

The idea is to combine four dimensions that each capture a different aspect of infrastructure.

1. `Segregation Quality` - How much cycling infrastructure is properly separated from motor traffic.

In [123]:
be_infrastructure['ratio_segregated'] = be_infrastructure['ratio-cycle_tracks-main_roads']

2. `Contraflow Access` - Can cyclists travel both directions on one-way streets?

In [124]:
be_infrastructure['ratio_contraflow'] = be_infrastructure['ratio_contraflow']

3. `Surface Quality` - Share of cycling infrastructure has poor or unknown surface quality?

In [125]:
quality_cols = [
    'percent_quality-track-badly_rideable',
    'percent_quality-track-not_rideable',
    'percent_quality-lane-badly_rideable',
    'percent_quality-lane-not_rideable',
    'percent_quality-shared_pedestrians-badly_rideable',
    'percent_quality-shared_pedestrians-not_rideable',
]
be_infrastructure['bad_quality'] = be_infrastructure[quality_cols].mean(axis=1).fillna(0)

4. `Surface Type` - Share of infrastructure that uses poor materials (gravel/dirt)

In [126]:
be_infrastructure['bad_surface'] = (
    be_infrastructure['percent_surface_type-track-gravel_dirt'].fillna(0) +
    be_infrastructure['percent_surface_type-shared_pedestrians-gravel_dirt'].fillna(0)
) / 2

Creating the **`Infrastructure Deficit`** Metric:

In [127]:
from sklearn.preprocessing import MinMaxScaler
seg_scaler = MinMaxScaler()
contra_scaler = MinMaxScaler()

be_infrastructure['seg_score'] = seg_scaler.fit_transform(be_infrastructure[['ratio_segregated']].fillna(0))
be_infrastructure['contra_score'] = contra_scaler.fit_transform(be_infrastructure[['ratio_contraflow']].fillna(0))
be_infrastructure['quality_score'] = be_infrastructure['bad_quality'] / 100
be_infrastructure['surface_score'] = be_infrastructure['bad_surface'] / 100

# Setting Equal Weights to begin with
be_infrastructure['infrastructure_deficit'] = (
    0.25 * (1 - be_infrastructure['seg_score']) +      # lack of segregated infra
    0.25 * (1 - be_infrastructure['contra_score']) +   # lack of contraflow
    0.25 * be_infrastructure['quality_score'] +        # poor rideability
    0.25 * be_infrastructure['surface_score']          # poor surface material
)

In [128]:
be_infrastructure.loc[be_infrastructure['infrastructure_deficit'].idxmax()]

Country                                    BE
City                              Arr. Veurne
NUTS_ID                                 BE258
Lat, Lon                  51.035332, 2.661147
Area                                   279.28
                                 ...         
seg_score                            0.017794
contra_score                         0.440507
quality_score                             0.0
surface_score                         0.09799
infrastructure_deficit               0.409922
Name: 86, Length: 151, dtype: object

In [129]:
be_infrastructure.loc[be_infrastructure['City'] == 'Arr. Leuven', 'infrastructure_deficit']

65    0.19389
Name: infrastructure_deficit, dtype: float64

Saving the Infrastructure Deficit CSV:

In [130]:
be_infrastructure.to_csv('be_infrastructure.csv', index=False)